In [ ]:

## Classes Base

##Utilizamos como base a classe `Graph` fornecida nos exercícios da Semana 7.

In [ ]:
class Vertex:
    '''Estrutura de Vértice para um grafo'''
    def __init__(self, vertex_id):
        self._vertex_id = vertex_id

    def __hash__(self):
        return hash(self._vertex_id)

    def __str__(self):
        return 'v{0}'.format(self._vertex_id)

    def __eq__(self, vertex):
        return self._vertex_id == vertex._vertex_id

    def __lt__(self, vertex):
        return self._vertex_id < vertex._vertex_id

    def __le__(self, vertex):
        return self._vertex_id <= vertex._vertex_id

    def __gt__(self, vertex):
        return self._vertex_id > vertex._vertex_id

    def __ge__(self, vertex):
        return self._vertex_id >= vertex._vertex_id

    def vertex_id(self):
        return self._vertex_id


class Edge:
    '''Estrutura de Aresta para um Grafo: (origem, destino) e peso'''
    def __init__(self, vertex_1, vertex_2, weight):
        self._vertex_1 = vertex_1
        self._vertex_2 = vertex_2
        self._weight = weight

    def __hash__(self):
        return hash((self._vertex_1, self._vertex_2))

    def __str__(self):
        return 'e({0},{1})w={2}'.format(self._vertex_1, self._vertex_2, self._weight)

    def __eq__(self, other):
        return self._vertex_1 == other._vertex_1 and self._vertex_2 == other._vertex_2

    def endpoints(self):
        return (self._vertex_1, self._vertex_2)

    def cost(self):
        return self._weight

    def opposite(self, vertex):
        if vertex == self._vertex_1:
            return self._vertex_2
        elif vertex == self._vertex_2:
            return self._vertex_1
        else:
            return None


class Graph:
    '''Representação de um grafo não orientado usando dicionários encadeados'''
    def __init__(self):
        self._adjancencies = {}
        self._vertices = {}
        self._n = 0
        self._m = 0

    def __str__(self):
        if self._n == 0:
            return "DAA-Graph: <empty>\n"
        ret = "DAA-Graph:\n"
        for vertex in self._adjancencies.keys():
            ret += str(vertex) + ": "
            for edge in self.incident_edges(vertex.vertex_id()):
                ret += str(edge) + "; "
            ret += "\n"
        return ret

    def is_directed(self):
        return False

    def order(self):
        return self._n

    def size(self):
        return self._m

    def has_vertex(self, vertex_id):
        return vertex_id in self._vertices

    def has_edge(self, u_id, v_id):
        if not self.has_vertex(u_id) or not self.has_vertex(v_id):
            return False
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        return vertex_v in self._adjancencies[vertex_u]

    def insert_vertex(self, vertex_id):
        if not self.has_vertex(vertex_id):
            vertex = Vertex(vertex_id)
            self._vertices[vertex_id] = vertex
            self._adjancencies[vertex] = {}
            self._n += 1

    def insert_edge(self, u_id, v_id, weight=0):
        if not self.has_vertex(u_id):
            self.insert_vertex(u_id)
        if not self.has_vertex(v_id):
            self.insert_vertex(v_id)
        if not self.has_edge(u_id, v_id):
            self._m += 1
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        e = Edge(vertex_u, vertex_v, weight)
        self._adjancencies[vertex_u][vertex_v] = e
        self._adjancencies[vertex_v][vertex_u] = e

    def degree(self, vertex_id):
        return len(self._adjancencies[self._vertices[vertex_id]])

    def get_vertex(self, vertex_id):
        return None if not self.has_vertex(vertex_id) else self._vertices[vertex_id]

    def get_edge(self, u_id, v_id):
        if not self.has_edge(u_id, v_id):
            return None
        vertex_u = self._vertices[u_id]
        vertex_v = self._vertices[v_id]
        return self._adjancencies[vertex_u][vertex_v]

    def vertices(self):
        return self._vertices.values()

    def edges(self):
        seen = {}
        for adj_map in self._adjancencies.values():
            for edge in adj_map.values():
                if edge not in seen:
                    yield edge
                seen[edge] = True

    def incident_edges(self, vertex_id):
        vertex = self._vertices[vertex_id]
        for edge in self._adjancencies[vertex].values():
            yield edge

    def has_neighbors(self, vertex_id):
        if not self.has_vertex(vertex_id):
            return False
        return self.degree(vertex_id) == 0

    def remove_vertex(self, vertex_id):
        if self.has_vertex(vertex_id):
            lst_copied = list(self.incident_edges(vertex_id))
            for edge in lst_copied:
                x, y = edge.endpoints()
                self.remove_edge(x.vertex_id(), y.vertex_id())
            del self._adjancencies[self._vertices[vertex_id]]
            del self._vertices[vertex_id]
            self._n -= 1

    def remove_edge(self, u_id, v_id):
        if self.has_edge(u_id, v_id):
            vertex_u = self._vertices[u_id]
            vertex_v = self._vertices[v_id]
            del self._adjancencies[vertex_u][vertex_v]
            if vertex_u != vertex_v:
                del self._adjancencies[vertex_v][vertex_u]
            self._m -= 1

In [ ]:
import csv
import os

def load_graph_csv(filepath):
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"File {filepath} not found.")
    
    graph = Graph()
    with open(filepath, mode='r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            # O .strip() previne que "Aemon " e "Aemon" sejam registados como dois vértices diferentes.
            source = row['Source'].strip()
            target = row['Target'].strip()
            weight = int(row['Weight'])
            graph.insert_edge(source, target, weight)

    return graph
    

In [ ]:

import csv
import os
from collections import deque
import matplotlib.pyplot as plt
class CentralityAnalyser:
    def __init__(self, graph):
        self._graph = graph
        self._vertex_ids = [v.vertex_id() for v in graph.vertices()]
        self._n = graph.order()
        self._m = graph.size()
    #Análise Teorica
    #2.1
    def bfs(self, source):
        if not self._graph.has_vertex(source):
            raise ValueError(f"Source vertex {source} not found in graph.")
        
        distances = {source: 0}

        queue = deque([source])
        while queue:
            current_id = queue.popleft()
            current_obj=self._graph.get_vertex(current_id)
            for edge in self._graph.incident_edges(current_id):
                neighbor_obj = edge.opposite(current_obj)
                neighbor_id = neighbor_obj.vertex_id()

                if neighbor_id not in distances:
                    distances[neighbor_id] = distances[current_id] + 1
                    queue.append(neighbor_id)

        return distances
    #2.2
    def num_components(self):
        count=0
        visited = set()

        for vertex in self._graph.vertices():
            if vertex.vertex_id() not in visited:
                count += 1
                reachable = self.bfs(vertex.vertex_id())
                visited.update(reachable.keys())
        
        return count

    def largest_component(self):
        visited = set()
        largest_size = 0
        largest_component_vertices = []

        for vertex in self._graph.vertices():
            if vertex.vertex_id() not in visited:
                reachable = self.bfs(vertex.vertex_id())
                visited.update(reachable.keys())
                component_size = len(reachable)

                if component_size > largest_size:
                    largest_size = component_size
                    largest_component_vertices = list(reachable.keys())

        return largest_component_vertices

    #2.3
    def degree_distribution(self):
        degree_count = {}
        for vertex in self._graph.vertices():
            degree = self._graph.degree(vertex.vertex_id())
            degree_count[degree] = degree_count.get(degree, 0) + 1
        return degree_count

    #2.4
    def diameter(self):
        largest_comp = self.largest_component()
        max_diameter = 0
        best_pair = (None, None)
        
        for vertex_obj in largest_comp.vertices():
            source_id = vertex_obj.vertex_id()
            distances = self.bfs(source_id)
            
            if distances:
                farthest_node = max(distances, key=distances.get)
                current_max = distances[farthest_node]
                
                if current_max > max_diameter:
                    max_diameter = current_max
                    best_pair = (source_id, farthest_node)
                    
        return max_diameter, best_pair

    #3.1
    def degree_centrality(self):

    #3.2
    def closeness_centrality(self):
    #3.3
    def eigenvector_centrality(self, epsilon: float = 1e-6, max_iter: int = 100):
    #3.4
    def betweenness_centrality(self):


IndentationError: expected an indented block after function definition on line 24 (3034405533.py, line 25)

1.2 Explicação:


## 2. Análise Estrutural dos Grafos
### 2.1 BFS
### 2.2 Conectividade
### 2.3 Distribuição de graus e top-10
### 2.4 Diâmetro

## 3. Métricas de Centralidade
### 3.1 Degree Centrality
### 3.2 Closeness Centrality
### 3.3 Eigenvector Centrality
### 3.4 Betweenness Centrality

## 4. Análise Empírica do Tempo de Execução
### 4.1 Escalabilidade
### 4.2 Comparação CC vs BC
### 4.3 Verificação TCC/TEC ≈ n/k

## 5. Questões Éticas
### 5.1 Colaborações externas
### 5.2 Referências